# Figure PS01 — Phage: the minus-strand label error and the shortcut it taught the model

**Status:** draft (Setonix integrator, agent-handoffs#24). Provisional ID; supplementary.

**Caption (draft).** **(a)** Six-frame ORFs ≥90 aa in 36,025 Inphared phage genomes (6,754,361 ORFs) labelled as matching an annotated CDS by the legacy matcher (before genome_entropy commit 2cc0e1a) and by the corrected matcher (v2), by strand. The legacy matcher compared reverse-complement ORF coordinates with genomic CDS coordinates and so labelled 3 of 3,311,070 minus-strand ORFs as genes; the corrected matcher labels 1,000,839. **(b)** Mean gain (5 grouped cross-validation folds; whiskers min–max) of each feature in the original 14-feature ranking model (31,200 genomes, 5,887,690 ORFs), trained on legacy or on corrected labels. On legacy labels the model learned "plus strand = gene" (`strand_plus` carries 0.89 of the gain); on corrected labels 3Di entropy dominates and strand is irrelevant. All phage results in the paper use the corrected labels.

**Manuscript text:** Methods, phage labels; Results, phage section.

## Provenance

| item | value |
|---|---|
| plotting data | `../data/figPS01_label_audit.tsv.gz`, `../data/figPS01_feature_importance.tsv.gz`; sha256 in `../data/phage_provenance.json` |
| produced by | `manuscript/prep/phage_prepare_data.py` |
| upstream | `Validation/mutual_information/tables/label_legacy_vs_v2.tsv` (#108, `scripts/compare_labels.py`); `Validation/Inphared/tables/model_importance.tsv` (#110, `scripts/03_models.py`) |
| populations | (a) the mutual-information ORF set; (b) the ranking ORF set (49 of 361 Inphared parts lack entropy JSON) — see `methods_provenance.md` |

## Data dictionary

| file | columns | meaning |
|---|---|---|
| `figPS01_label_audit` | `strand`, `contig_edge`, `legacy`, `v2`, `orfs` | ORF counts by label agreement |
| `figPS01_feature_importance` | `set` (`orig14`), `labels` (`json` = legacy, `relabelled` = corrected), `feature`, `gain_mean/min/max` | XGBoost gain over 5 folds |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import msfig

msfig.style()
SLUG = "figPS01_phage_label_correction"
C = msfig.PALETTE

In [ ]:
audit = msfig.data("figPS01_label_audit.tsv.gz")
imp = msfig.data("figPS01_feature_importance.tsv.gz")

pos = {(s, lab): audit[(audit.strand == s) & audit[lab]].orfs.sum()
       for s in "+-" for lab in ("legacy", "v2")}
assert pos["-", "legacy"] == 3 and pos["-", "v2"] == 1_000_839
assert audit[audit.strand == "-"].orfs.sum() == 3_311_070
assert audit.orfs.sum() == 6_754_361
o14 = imp[imp.set == "orig14"].pivot(index="feature", columns="labels", values="gain_mean")
assert round(o14.loc["strand_plus", "json"], 2) == 0.89

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), gridspec_kw={"width_ratios": [0.8, 1.2]})

ax = axes[0]
x = np.arange(2)
w = 0.38
for k, (lab, name, col) in enumerate([("legacy", "legacy matcher", C["grey"]),
                                      ("v2", "corrected matcher", C["blue"])]):
    v = [pos["+", lab] / 1e6, pos["-", lab] / 1e6]
    ax.bar(x + (k - 0.5) * w, v, w, color=col, label=name)
    for xx, n in zip(x + (k - 0.5) * w, [pos["+", lab], pos["-", lab]]):
        ax.text(xx, n / 1e6 + 0.03, f"{n:,}", ha="center", fontsize=6, rotation=90, va="bottom")
ax.set_xticks(x, ["plus strand", "minus strand"])
ax.set_ylabel("ORFs labelled as genes (millions)")
ax.set_ylim(0, 2.5)
ax.legend(loc="upper right", fontsize=6.5)
ax.set_title("a  labels by strand", loc="left", fontweight="bold")

ax = axes[1]
g = imp[imp.set == "orig14"]
order = (g[g.labels == "json"].sort_values("gain_mean").feature.tolist())
y = np.arange(len(order))
h = 0.38
for k, (lab, name, col) in enumerate([("json", "legacy labels", C["grey"]),
                                      ("relabelled", "corrected labels", C["blue"])]):
    d = g[g.labels == lab].set_index("feature").loc[order]
    ax.barh(y + (k - 0.5) * h, d.gain_mean, h, color=col, label=name,
            xerr=[d.gain_mean - d.gain_min, d.gain_max - d.gain_mean],
            error_kw=dict(lw=0.6, capsize=1))
ax.set_yticks(y, [f.replace("_", " ").replace("three di", "3Di") for f in order], fontsize=6.5)
ax.set_xlabel("feature gain (share of total)")
ax.set_xlim(0, 1)
ax.legend(loc="lower right", fontsize=6.5)
ax.set_title("b  what the 14-feature model learned", loc="left", fontweight="bold")

fig.tight_layout()
msfig.save(fig, SLUG)